# CWE Confusion Analysis

This notebook:
1. Loads the confusion matrix and maps numeric indices → CWE names
2. Extracts **only** misclassified pairs (off-diagonal)
3. Computes `Confusion_Rate` = misclassifications / total samples for that class
4. Filters to meaningful confusions and exports results

## 1. Load Confusion Matrix & Map to CWE Names

> **Note:** Use `confusion_matrix_cwe.csv` (CWE-named labels) if available.  
> If you only have `confusion_matrix.csv` (numeric indices), set `USE_NUMERIC_CM = True` below.

In [1]:
import pandas as pd
import numpy as np

# ─── CONFIG ──────────────────────────────────────────────────────────────────
# Set to True  → reads confusion_matrix.csv (numeric indices) + juliet_cwe_dataset.csv for mapping
# Set to False → reads confusion_matrix_cwe.csv (already has CWE names as index/columns)
USE_NUMERIC_CM = True

CM_FILE         = "confusion_matrix.csv"       # numeric-index confusion matrix
CM_CWE_FILE     = "confusion_matrix_cwe.csv"   # CWE-named confusion matrix
MAPPING_FILE    = "juliet_cwe_dataset.csv"      # source dataset with cwe_index + cwe columns
# ─────────────────────────────────────────────────────────────────────────────

if USE_NUMERIC_CM:
    cm = pd.read_csv(CM_FILE, index_col=0)

    # Ensure index/columns are integers so the mapping lookup works
    cm.index   = cm.index.astype(int)
    cm.columns = cm.columns.astype(int)

    # Build index → CWE name mapping from the source dataset
    mapping_df = (
        pd.read_csv(MAPPING_FILE)[["cwe_index", "cwe"]]
        .drop_duplicates()
    )
    cwe_map = dict(zip(mapping_df["cwe_index"], mapping_df["cwe"]))

    # Replace numeric index/columns with CWE names
    cm.index   = cm.index.map(cwe_map)
    cm.columns = cm.columns.map(cwe_map)

    print(f"✅ Loaded numeric CM and mapped to CWE names. Shape: {cm.shape}")

else:
    cm = pd.read_csv(CM_CWE_FILE, index_col=0)
    print(f"✅ Loaded CWE-named CM. Shape: {cm.shape}")

cm.head(3)

✅ Loaded numeric CM and mapped to CWE names. Shape: (56, 56)


,CWE114,CWE121,CWE122,CWE123,CWE124,CWE126,CWE127,CWE134,CWE15,CWE176,...,CWE397,CWE398,CWE400,CWE401,CWE404,CWE415,CWE416,CWE426,CWE427,CWE440
CWE114,440,0,0,0,0,0,4,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CWE121,0,2772,0,6,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CWE122,0,0,3606,6,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 2. Extract Off-Diagonal Misclassifications

`Confusion_Rate` here means the **error rate** for a given true class:  
$$\text{Confusion Rate} = \frac{\text{misclassified as } \hat{y}}{\text{total samples of true class } y}$$

This is *different* from accuracy — a value of `0.003` means 0.3% of that class was wrongly predicted as the other.

In [2]:
confusions = []

for true_cwe in cm.index:
    total_samples = cm.loc[true_cwe].sum()
    if total_samples == 0:
        continue

    for pred_cwe in cm.columns:
        # ✅ FIX: skip DIAGONAL (correct predictions) — we only want errors
        if true_cwe == pred_cwe:
            continue

        count = cm.loc[true_cwe, pred_cwe]

        if count > 0:
            confusion_rate = count / total_samples   # error rate for this class
            confusions.append({
                "True_CWE":      true_cwe,
                "Predicted_CWE": pred_cwe,
                "Count":         int(count),
                "Total_Samples": int(total_samples),
                "Confusion_Rate": round(confusion_rate, 6)
            })

confusions_df = (
    pd.DataFrame(confusions)
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

print(f"Total misclassified pairs found: {len(confusions_df)}")
print(f"Total misclassified samples    : {confusions_df['Count'].sum()}")
confusions_df.head(20)

Total misclassified pairs found: 16
Total misclassified samples    : 56


,True_CWE,Predicted_CWE,Count,Total_Samples,Confusion_Rate
0,CWE188,CWE190,10,740,0.013514
1,CWE176,CWE190,8,740,0.010811
2,CWE121,CWE123,6,2780,0.002158
3,CWE122,CWE123,6,3614,0.001660
4,CWE114,CWE127,4,444,0.009009
5,CWE121,CWE190,2,2780,0.000719
6,CWE122,CWE190,2,3614,0.000553
7,CWE123,CWE15,2,1598,0.001252
8,CWE124,CWE190,2,950,0.002105
9,CWE126,CWE15,2,1598,0.001252


## 3. Top Confused Pairs (human-readable)

Sorted by raw count — the most frequent mistakes your model makes.

In [3]:
print("Top confused CWE pairs:")
for _, row in confusions_df.head(20).iterrows():
    print(
        f"  {row['True_CWE']:>10}  →  {row['Predicted_CWE']:<10}"
        f"  {row['Count']:>4} times  "
        f"({row['Confusion_Rate']*100:.2f}% of {row['Total_Samples']} samples)"
    )

Top confused CWE pairs:
      CWE188  →  CWE190        10 times  (1.35% of 740 samples)
      CWE176  →  CWE190         8 times  (1.08% of 740 samples)
      CWE121  →  CWE123         6 times  (0.22% of 2780 samples)
      CWE122  →  CWE123         6 times  (0.17% of 3614 samples)
      CWE114  →  CWE127         4 times  (0.90% of 444 samples)
      CWE121  →  CWE190         2 times  (0.07% of 2780 samples)
      CWE122  →  CWE190         2 times  (0.06% of 3614 samples)
      CWE123  →  CWE15          2 times  (0.13% of 1598 samples)
      CWE124  →  CWE190         2 times  (0.21% of 950 samples)
      CWE126  →  CWE15          2 times  (0.13% of 1598 samples)
      CWE134  →  CWE15          2 times  (0.06% of 3104 samples)
      CWE134  →  CWE190         2 times  (0.06% of 3104 samples)
       CWE15  →  CWE190         2 times  (0.12% of 1736 samples)
      CWE176  →  CWE15          2 times  (0.27% of 740 samples)
      CWE188  →  CWE15          2 times  (0.27% of 740 samples)
      C

## 4. Filter: Critical Confusions

Keep only pairs where the **error rate exceeds a threshold** (default 1% = 0.01).  
These are classes that meaningfully struggle against a specific other class.

In [4]:
# ✅ FIX: threshold now makes sense — 0.01 = at least 1% of samples misclassified
# (The old notebook used 0.05 on diagonal-included data, which was misleading)
CONFUSION_RATE_THRESHOLD = 0.01

critical_confusions = (
    confusions_df[confusions_df["Confusion_Rate"] >= CONFUSION_RATE_THRESHOLD]
    .reset_index(drop=True)
)

print(f"Critical confusion pairs (error rate ≥ {CONFUSION_RATE_THRESHOLD*100:.0f}%): {len(critical_confusions)}")
print()
print(critical_confusions.to_string(index=False))

Critical confusion pairs (error rate ≥ 1%): 2

True_CWE Predicted_CWE  Count  Total_Samples  Confusion_Rate
  CWE188        CWE190     10            740        0.013514
  CWE176        CWE190      8            740        0.010811


## 5. Semantic Group Analysis

Tag each CWE with a semantic group to see if confusions happen *within* related vulnerability families.

In [5]:
similar_groups = {
    "Buffer Issues":    ["CWE121", "CWE122", "CWE124", "CWE123", "CWE126", "CWE127"],
    "Integer Errors":   ["CWE190", "CWE191", "CWE194", "CWE195", "CWE188", "CWE196"],
    "Resource Errors":  ["CWE404", "CWE415", "CWE416", "CWE401"],
    "Path Traversal":   ["CWE23",  "CWE36"],
    "Format String":    ["CWE134"],
    "Memory Issues":    ["CWE457", "CWE476", "CWE590"],
}

def find_group(cwe):
    for group, members in similar_groups.items():
        if cwe in members:
            return group
    return "Other"

confusions_df["True_Group"] = confusions_df["True_CWE"].apply(find_group)
confusions_df["Pred_Group"] = confusions_df["Predicted_CWE"].apply(find_group)

# Confusions WITHIN the same semantic group
intra_group = confusions_df[
    (confusions_df["True_Group"] == confusions_df["Pred_Group"]) &
    (confusions_df["True_Group"] != "Other")
].reset_index(drop=True)

print(f"Intra-group confusions: {len(intra_group)}")
print(intra_group[["True_CWE", "Predicted_CWE", "Count", "Confusion_Rate", "True_Group"]].to_string(index=False))

Intra-group confusions: 3
True_CWE Predicted_CWE  Count  Confusion_Rate     True_Group
  CWE188        CWE190     10        0.013514 Integer Errors
  CWE121        CWE123      6        0.002158  Buffer Issues
  CWE122        CWE123      6        0.001660  Buffer Issues


## 6. Worst Performing Classes

Which true classes had the most total misclassifications?

In [6]:
worst = (
    confusions_df
    .groupby("True_CWE")["Count"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"Count": "Total_Errors"})
)

# Add total samples per class (diagonal value)
diag_counts = {cwe: int(cm.loc[cwe, cwe]) for cwe in cm.index if cwe in cm.columns}
worst["Correct"]       = worst["True_CWE"].map(diag_counts)
worst["Total_Samples"] = worst["Total_Errors"] + worst["Correct"]
worst["Error_Rate"]    = (worst["Total_Errors"] / worst["Total_Samples"]).round(4)

print("Worst performing CWE classes:")
print(worst.head(20).to_string(index=False))

Worst performing CWE classes:
True_CWE  Total_Errors  Correct  Total_Samples  Error_Rate
  CWE188            12      728            740      0.0162
  CWE176            10      730            740      0.0135
  CWE121             8     2772           2780      0.0029
  CWE122             8     3606           3614      0.0022
  CWE114             4      440            444      0.0090
  CWE134             4     3100           3104      0.0013
  CWE123             2     1596           1598      0.0013
  CWE124             2      948            950      0.0021
  CWE126             2     1596           1598      0.0013
   CWE15             2     1734           1736      0.0012
  CWE244             2      454            456      0.0044


## 7. Export Results

In [7]:
# All misclassified pairs
confusions_df.to_csv("cwe_confusion_analysis.csv", index=False)

# Only critical ones
critical_confusions.to_csv("cwe_critical_confusions.csv", index=False)

# Worst classes
worst.to_csv("cwe_worst_classes.csv", index=False)

print("✅ Exported:")
print("   cwe_confusion_analysis.csv   — all misclassified pairs")
print("   cwe_critical_confusions.csv  — pairs above error-rate threshold")
print("   cwe_worst_classes.csv        — per-class error summary")

✅ Exported:
   cwe_confusion_analysis.csv   — all misclassified pairs
   cwe_critical_confusions.csv  — pairs above error-rate threshold
   cwe_worst_classes.csv        — per-class error summary
